<div align="right"><a href="S12-Filter-Structures.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L12-Filterstrukturen.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L11 Korrelationsfolgen](./L11-Korrelationsfolgen.ipynb) | [Aufgabe 12](./A12-Filterstrukturen.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L13 Entwurf rekursiver Filter](./L13-Entwurf-rekursiver-Filter.ipynb)

</div>

---


Aufgabe: Strukturen von Filtern

Bis jetzt wurden LTI-Systeme durch die Koeffizienten der Vektoren `b` und `a` beschrieben, was der üblichen Darstellungsform mittels System- oder Z-Übertragungsfunktion entspricht:

$$
H(z) = \frac{b_0 + b_1 z^{-1} + b_2 z^{-2} + \cdots}{1+ a_1 z^{-1} + a_2 z^{-2} + \cdots} \,\,\,.
$$

Eine andere (gleichwertige) Beschreibung kann mit Polen und Nullstellen erfolgen:

$$
H(z) = b_0 \frac{(z-z_{0,1})(z-z_{0,2})(z-z_{0,3})\cdots}{(z-z_{\infty,1})(z-z_{\infty,2})(z-z_{\infty,3})\cdots} \qquad.
$$

LTI-Systeme können mit *kanonischen* Strukturen implementiert werden. Eine kanonische Struktur ist in der Regel eine Realisierungsform mit minimaler Anzahl an Speicherelementen. Siehe Kapitel 4.1 für die Beschreibung der jeweiligen Implementierung. Die in dieser Aufgabe betrachtete *dritte kanonische* Struktur erlaubt es, die Systemfunktion direkt in eine Pol-Nullstellen-Form zu überführen.

Die einzelnen Pole bzw. Nullstellen werden in Abschnitte maximal zweiter Ordnung gruppiert (SOS = Second Order Section) und in einem Netzwerk kaskadiert. Die Hauptvorteile dieser Struktur sind die Flexibilität der Anordnung sowie ein gutes Verhalten bei der Verwendung quantisierter Parameter.

(Hinweis: Zum Thema Parameterquantisierung siehe Abschnitt 4.4 bzw. Aufgabe dsv4a04e.)

Die MATLAB-Funktion `tf2zp()` konvertiert die Polynomdarstellung in die Pol-Nullstellen-Darstellung. Mit dieser Darstellung kann die MATLAB-Funktion `zp2sos()` die Second-Order-Sections der dritten kanonischen Form berechnen.

---

a)

Schreiben Sie eine neue MATLAB-Funktion `[B, A] = lcascade(b, a)`, die aus den Filtervektoren `b` und `a` Abschnitte zweiter Ordnung bildet. Die Koeffizienten der Second-Order-Sections sollen mit `tf2zp()` und `zp2sos()` berechnet und als Zeilenvektoren in den Matrizen `B` und `A` gespeichert werden.

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ellip, freqz
from scipy.signal import tf2zpk, zpk2sos


def lcascade(b, a):
    z, p, k = tf2zpk(b, a)
    sos = zpk2sos(z, p, k)
    B = sos[:, :3]
    A = sos[:, 3:6]
    
    return B, A 
# END SOLUTION

b)

Berechnen Sie (mithilfe von `lcascade()`) die Filterkoeffizienten der Second-Order-Sections eines Cauer-Tiefpasses 7. Ordnung mit folgenden Spezifikationen:

- $3\,\text{dB}$-Grenzfrequenz: $0{,}4\,\pi$
- Dämpfung im Durchlassbereich: $R_P = 0{,}1\,\text{dB}$
- Dämpfung im Sperrbereich: $R_S = 40\,\text{dB}$

Die Toleranzparameter $R_S$ bzw. $R_P$ sind wie folgt definiert:

$$
    R_P = -20 \log_{10}(1 - \delta_D), \quad
    R_S = -20 \log_{10}(\delta_S),
$$
wobei $\delta_D$ bzw. $\delta_S$ die Toleranzparameter gemäß Kapitel 4 darstellen.

Die MATLAB-Funktion `ellip(7, 0.1, 40, 0.4)` liefert die Filterkoeffizienten:
$$
    [b, a] = ellip(7, 0.1, 40, 0.4);
    [B, A] = lcascade(b, a);
$$
Berechnen Sie die Übertragungsfunktion des Gesamtsystems gemäß:
$$
    H(z) = \prod_{i=1}^{p} H_i(z).
$$

In [ ]:
# BEGIN SOLUTION
b, a = ellip(7, 0.1, 40, 0.4)  # Entwurf des Filters

B, A = lcascade(b, a)  # Umwandlung in die 3. kanonische Struktur

m, n = B.shape
Hi = np.zeros((512, m), dtype=complex)

# Frequenzachse definieren (gemeinsam für alle Teilsysteme)
W = np.linspace(0, np.pi, 512)

# Übertragungsfunktion des Gesamtsystems auf fester Frequenzachse berechnen
_, Hg = freqz(b, a, 512)

# Übertragungsfunktionen der Teilsysteme ebenfalls auf fester Frequenzachse berechnen
Hi = np.zeros((512, m), dtype=complex)
for k in range(m):
    _, H_temp = freqz(B[k, :], A[k, :], worN=W)
    Hi[:, k] = H_temp

# Produkt der Frequenzgänge aller Teilsysteme
H3k = np.prod(Hi, axis=1)

# Darstellung der Frequenzgänge
plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H3k)), '--b', label='Produkt der Teilsysteme')
plt.plot(W / np.pi, 20 * np.log10(np.abs(Hg)), '-.b', label='Gesamtsystem')
plt.grid()
plt.xlabel('Omega/pi (Nyquist = 1)')
plt.ylabel('|H(exp(j*Omega))| in dB')
plt.title('-- Gesamtsystem, -.Produkt der Teilsysteme')
plt.ylim([-100, None])
plt.legend()
plt.show()
#END SOLUTION